# 02. Exploratory Data Analysis & Business Insights
**Author:** Renaldy Bilal Setyawan | **Stack:** DuckDB, JupySQL, Python (Pandas)  
**Dataset:** Gayanara E-Commerce (Cleaned)

---

### 📌 Overview
Building on the standardized database from **Phase 1**, this notebook executes **Phase 2 (Exploratory Data Analysis)**. 
Here, we translate raw transactional data into actionable business intelligence, focusing on revenue trends, product performance, and customer purchasing behavior.

In [2]:
import duckdb

# Connect to our existing, cleaned database
con = duckdb.connect('gayanara.db')

# Load the SQL extension and bind the connection
%load_ext sql
%sql con
%config SqlMagic.displaylimit = 50

## 📈 Task 1: Macro Business Metrics (Revenue Trends)
**Business Question:** What is our total Gross Merchandise Value (GMV) and how is our order volume trending over time?

In [9]:
%%sql
SELECT 
    date_trunc('month', order_date) as months,
    COUNT(DISTINCT o.order_id) AS total_completed_orders,
    SUM(p.price_idr) AS total_revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
GROUP BY months
ORDER BY months ASC;

Running query in 'DuckDBPyConnection'

months,total_completed_orders,total_revenue
2022-01-01,48,20798000
2022-02-01,29,10863000
2022-03-01,45,19586000
2022-04-01,38,14714000
2022-05-01,39,16676000
2022-06-01,38,11580000
2022-07-01,38,15418000
2022-08-01,35,11799000
2022-09-01,34,14528000
2022-10-01,42,17067000


> 💡 **Key Business Insights (Revenue Trend):**
> * **Massive Growth:** Monthly Gross Merchandise Value (GMV) has scaled significantly, growing from an average of ~15,000,000 IDR per month in early 2022 to over 57,000,000 IDR by Q1 2025.
> * **Volume Increase:** Order volume mirrors this trend, jumping from ~40 orders/month to over 120 orders/month, indicating strong customer acquisition and market penetration.

## 🏆 Task 2: Product Performance (Best-Selling Categories)
**Business Question:** Which product categories drive the highest volume, and which drive the highest total revenue?

In [12]:
%%sql
SELECT 
    p.category_clean,
    COUNT(*) AS total_items_sold,
    SUM(p.price_idr) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category_clean
ORDER BY total_revenue DESC;

Running query in 'DuckDBPyConnection'

category_clean,total_items_sold,total_revenue
Pants,841,205989000
Accessories,893,205077000
Jacket,926,204704000
Shirt,755,200105000
Dress,780,195630000
T-Shirt,791,180939000


> 💡 **Key Business Insights (Category Performance):**
> * **Revenue vs. Volume Mismatch:** **Jackets** drive the highest sales volume (926 units), but **Pants** drive the highest overall revenue (205.9M IDR), indicating Pants have a higher average selling price.
> * **Catalog Health:** The revenue distribution is exceptionally balanced. The top three categories (Pants, Accessories, Jackets) all generate roughly ~205M IDR, meaning the business is well-diversified and not overly reliant on a single product line.

## 👥 Task 3: Customer Segmentation (RFM Base Metrics)
**Business Question:** Who are our most valuable customers based on Recency, Frequency, and Monetary (RFM) spending habits?

In [13]:
%%sql
SELECT 
    c.email,
    CAST((SELECT MAX(order_date) FROM orders) AS DATE) - CAST(MAX(o.order_date) AS DATE) AS recency_days,
    COUNT(DISTINCT o.order_id) AS frequency,
    SUM(o.total_amount_idr) AS monetary_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY o.customer_id, c.email
ORDER BY monetary_value DESC
LIMIT 15;

Running query in 'DuckDBPyConnection'

email,recency_days,frequency,monetary_value
dewihutapea591@yahoo.com,129,6,7803360
hadifauzi621@yahoo.com,5,7,6615921
yogaiskandar656@gmail.com,56,7,6108037
ekosaputra985@gmail.com,154,8,5907000
putrapermata695@outlook.com,9,10,5733033
anggimanurung116@gmail.com,27,8,5109045
hanasimbolon349@yahoo.com,229,3,5034443
dionsiregar730@gmail.com,19,8,4912003
vivilestari325@gmail.com,24,9,4865145
adityasiregar323@yahoo.com,39,6,4851218


> 💡 **Key Business Insights (RFM Baselines):**
> * **High-Value Churn Risk:** Our #1 highest-spending customer has not made a purchase in 129 days. Another top-10 spender hasn't purchased in 229 days.
> * **The Goal:** We need to move away from raw lists and segment these customers into actionable tiers (e.g., "Champions" vs. "At Risk") so the marketing team can run targeted re-engagement campaigns.

In [15]:
%%sql
WITH rfm_base AS (
    SELECT 
        c.email,
        CAST((SELECT MAX(order_date) FROM orders) AS DATE) - CAST(MAX(o.order_date) AS DATE) AS recency_days,
        COUNT(DISTINCT o.order_id) AS frequency,
        SUM(o.total_amount_idr) AS monetary_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.email
)
SELECT 
    email,
    recency_days,
    frequency, 
    monetary_value,
    CASE 
        WHEN recency_days <= 60 AND frequency >= 5 AND monetary_value >= 4000000 THEN '🏆 Champions'
        WHEN recency_days > 60 AND (frequency >= 5 OR monetary_value >= 4000000) THEN '⚠️ At Risk (VIPs)'
        WHEN recency_days <= 30 AND frequency < 3 THEN '👋 New / Promising'
        WHEN recency_days >= 120 THEN '💤 Lost / Inactive'
        ELSE '🛒 Regulars'
    END AS customer_segment
FROM rfm_base
ORDER BY monetary_value DESC
LIMIT 15;

Running query in 'DuckDBPyConnection'

email,recency_days,frequency,monetary_value,customer_segment
dewihutapea591@yahoo.com,129,6,7803360,⚠️ At Risk (VIPs)
hadifauzi621@yahoo.com,5,7,6615921,🏆 Champions
yogaiskandar656@gmail.com,56,7,6108037,🏆 Champions
ekosaputra985@gmail.com,154,8,5907000,⚠️ At Risk (VIPs)
putrapermata695@outlook.com,9,10,5733033,🏆 Champions
anggimanurung116@gmail.com,27,8,5109045,🏆 Champions
hanasimbolon349@yahoo.com,229,3,5034443,⚠️ At Risk (VIPs)
dionsiregar730@gmail.com,19,8,4912003,🏆 Champions
vivilestari325@gmail.com,24,9,4865145,🏆 Champions
adityasiregar323@yahoo.com,39,6,4851218,🏆 Champions


> 💡 **Key Business Insights (RFM Segmentation):**
> * **Strict Rule-Based Segmentation:** By applying hardcoded business thresholds (e.g., 60-day recency limits), the model now accurately reflects true churn risk rather than just relative percentiles. 
> * **Actionable Takeaway:** Top spenders who have gone dormant (like our #1 customer at 129 days without a purchase) are now correctly flagged as **⚠️ At Risk (VIPs)** instead of Champions. Marketing can instantly export this specific segment to run aggressive, highly-targeted win-back campaigns.

## 🗺️ Task 4: Geospatial Revenue Map & AOV (Hidden Gems)
**Business Question:** Which cities deserve more ad budget? Are there high-value markets outside the major metropolitan areas that yield a higher Average Order Value (AOV)?

In [16]:
%%sql
SELECT 
    shipping_province,
    shipping_city,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(total_amount_idr) AS total_revenue,
    CAST(SUM(total_amount_idr) / COUNT(DISTINCT order_id) AS BIGINT) AS average_order_value
FROM orders
WHERE order_status NOT IN ('cancelled', 'returned')
GROUP BY shipping_province, shipping_city
ORDER BY total_revenue DESC
LIMIT 15;

Running query in 'DuckDBPyConnection'

shipping_province,shipping_city,total_orders,total_revenue,average_order_value
Jawa Tengah,Semarang,141,76571616,543061
Jawa Barat,Depok,150,75974249,506495
Sumatera Utara,Medan,155,75476440,486945
Jawa Barat,Bekasi,151,69672371,461406
Sulawesi Utara,Manado,138,69199639,501447
Sulawesi Selatan,Makassar,138,67318630,487816
Kalimantan Barat,Pontianak,148,67128230,453569
Banten,Tangerang,132,66851296,506449
Jawa Timur,Surabaya,112,65661283,586261
Kalimantan Timur,Balikpapan,124,64958371,523858


> 💡 **Key Business Insights (Geospatial Analysis):**
> * **Top Revenue Driver:** Semarang (Jawa Tengah) leads the pack in total revenue (~76.5M IDR), closely followed by Depok and Medan.
> * **The "Hidden Gems":** **Surabaya (Jawa Timur)** and **Banjarmasin (Kalimantan Selatan)** are the standout markets for profitability. Despite having lower order volumes (112 and 105 respectively), they boast the highest Average Order Values in the top 15 (Surabaya at 586k IDR and Banjarmasin at 583k IDR).
> * **Actionable Takeaway:** Marketing should allocate specific, high-end product ad spend to Surabaya and Banjarmasin, as customers in these regions are clearly willing to add more expensive items (or larger quantities) to their carts.

## 💸 Task 5: Promo Code ROI Evaluation
**Business Question:** Do promo codes actually incentivize higher spending (higher AOV), or are we just giving away discounts to people who would have bought anyway?

In [17]:
%%sql
SELECT 
    CASE 
        WHEN promo_code IS NULL THEN 'No Promo' 
        ELSE 'Used Promo' 
    END AS promo_status,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(total_amount_idr) AS total_revenue,
    SUM(COALESCE(discount_amount_idr, 0)) AS total_discount_given,
    CAST(SUM(total_amount_idr) / COUNT(DISTINCT order_id) AS BIGINT) AS average_order_value
FROM orders
WHERE order_status NOT IN ('cancelled', 'returned')
GROUP BY promo_status;

Running query in 'DuckDBPyConnection'

promo_status,total_orders,total_revenue,total_discount_given,average_order_value
Used Promo,1025,496651379,31427621,484538
No Promo,1496,768689000,0,513830


> 💡 **Key Business Insights (Promo Code ROI):**
> * **Negative AOV Impact:** Orders utilizing promo codes actually have a lower Average Order Value (484k IDR) compared to organic, non-promo orders (513k IDR). 
> * **Burning Money:** The company has given away over 31.4M IDR in discounts without seeing a corresponding lift in cart sizes.
> * **Actionable Takeaway:** The current promo strategy is cannibalizing revenue. Marketing needs to pivot from flat discounts to "Threshold Promos" (e.g., "Get 50k off when you spend 600k") to force the promo AOV higher than the organic AOV.

## 🚚 Task 6: Courier Return & Cancellation Rates
**Business Question:** Which couriers have the highest return and cancellation rates? Is there a specific logistics partner causing fulfillment issues?

In [18]:
%%sql
SELECT 
    courier,
    COUNT(order_id) AS total_orders,
    SUM(CASE WHEN order_status = 'returned' THEN 1 ELSE 0 END) AS returned_orders,
    SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
    ROUND(SUM(CASE WHEN order_status = 'returned' THEN 1.0 ELSE 0 END) / COUNT(order_id) * 100, 2) AS return_rate_pct,
    ROUND(SUM(CASE WHEN order_status = 'cancelled' THEN 1.0 ELSE 0 END) / COUNT(order_id) * 100, 2) AS cancel_rate_pct
FROM orders
GROUP BY courier
ORDER BY return_rate_pct DESC;

Running query in 'DuckDBPyConnection'

courier,total_orders,returned_orders,cancelled_orders,return_rate_pct,cancel_rate_pct
Pos Indonesia,161,11,12,6.83,7.45
J&T,896,54,111,6.03,12.39
JNE,845,43,84,5.09,9.94
SiCepat,791,35,77,4.42,9.73
Anteraja,307,8,44,2.61,14.33


> 💡 **Key Business Insights (Courier Evaluation):**
> * **Highest Absolute Returns:** While Pos Indonesia has the highest return rate percentage (6.83%), **J&T** is the biggest operational bottleneck. They handle the most volume but still maintain a high return rate (6.03%), resulting in the most damaged/returned goods overall.
> * **The Cancellation Red Flag:** **Anteraja** has the lowest return rate (2.61%) but the highest cancellation rate (14.33%). This points to pre-delivery fulfillment issues, such as slow dispatch times or poor tracking updates causing customers to abandon their orders.
> * **Actionable Takeaway:** Renegotiate SLAs (Service Level Agreements) with J&T regarding package handling, and audit Anteraja's pickup times to reduce the pre-shipment cancellation spike.

## 🔄 Task 7: Cohort Retention Analysis
**Business Question:** How loyal are our customers? If a user makes their first purchase in a specific month, what percentage of those users return to buy again in the following months?

In [19]:
%%sql
WITH first_purchases AS (
    -- Step 1: Find the first month a customer ever made a purchase
    SELECT 
        customer_id,
        DATE_TRUNC('month', MIN(order_date)) AS cohort_month
    FROM orders
    GROUP BY customer_id
),
cohort_data AS (
    -- Step 2: Join back to orders to find all subsequent purchase months
    SELECT
        f.cohort_month,
        DATE_TRUNC('month', o.order_date) AS activity_month,
        o.customer_id
    FROM orders o
    JOIN first_purchases f ON o.customer_id = f.customer_id
),
cohort_sizes AS (
    -- Step 3: Count how many original customers were in that first cohort
    SELECT 
        cohort_month,
        COUNT(DISTINCT customer_id) AS initial_customers
    FROM first_purchases
    GROUP BY cohort_month
),
retention_counts AS (
    -- Step 4: Calculate the month index (Month 0, Month 1, etc.) and count active users
    SELECT
        c.cohort_month,
        DATE_DIFF('month', c.cohort_month, c.activity_month) AS month_index,
        COUNT(DISTINCT c.customer_id) AS active_customers
    FROM cohort_data c
    GROUP BY c.cohort_month, month_index
)
-- Step 5: Put it all together and calculate the retention percentage
SELECT
    STRFTIME(r.cohort_month, '%Y-%m') AS cohort_month,
    s.initial_customers,
    r.month_index,
    r.active_customers,
    ROUND((r.active_customers * 100.0) / s.initial_customers, 2) AS retention_pct
FROM retention_counts r
JOIN cohort_sizes s ON r.cohort_month = s.cohort_month
ORDER BY r.cohort_month ASC, r.month_index ASC
LIMIT 30;

Running query in 'DuckDBPyConnection'

cohort_month,initial_customers,month_index,active_customers,retention_pct
2022-01,47,0,47,100.0
2022-01,47,1,2,4.26
2022-01,47,2,3,6.38
2022-01,47,3,2,4.26
2022-01,47,4,1,2.13
2022-01,47,5,2,4.26
2022-01,47,6,4,8.51
2022-01,47,7,4,8.51
2022-01,47,9,3,6.38
2022-01,47,11,5,10.64


> 💡 **Key Business Insights (Cohort Retention):**
> * **Severe Early Churn:** Early cohort data (e.g., Jan 2022) shows incredibly low early-stage retention, with only 4.26% of customers returning in Month 1 and 6.38% in Month 2. 
> * **Event-Driven Reactivation:** There are occasional late-stage spikes (e.g., 21.28% retention in Month 14), suggesting old customers only return when incentivized by massive seasonal sales or marketing events.
> * **Actionable Takeaway:** The business is bleeding new users. Marketing must prioritize an automated "welcome series" and an aggressive 30-day win-back email sequence to hook customers immediately after their first purchase.

## 📤 Data Export for Executive Dashboard (Phase 3)
**Objective:** To ensure high performance in the final Business Intelligence (BI) dashboard, we are exporting the aggregated insights from our Exploratory Data Analysis. Connecting a BI tool directly to pre-calculated CSVs prevents the visualization engine from lagging when processing thousands of raw transaction rows.

The following dataframes are exported for visualization:
1. **Geospatial Revenue:** To map out top-performing cities and identify high-AOV "hidden gems."
2. **Promo ROI:** To visualize the negative AOV impact of the current discount strategy.
3. **Courier Evaluation:** To chart the return and cancellation rates for logistics auditing.
4. **RFM Segments:** To display the distribution of our customer base across active, at-risk, and churned tiers.

In [20]:
%%sql
-- 1. Exporting Geospatial Data
COPY (
    SELECT 
        shipping_province,
        shipping_city,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(total_amount_idr) AS total_revenue,
        CAST(SUM(total_amount_idr) / COUNT(DISTINCT order_id) AS BIGINT) AS average_order_value
    FROM orders
    WHERE order_status NOT IN ('cancelled', 'returned')
    GROUP BY shipping_province, shipping_city
    ORDER BY total_revenue DESC
) TO 'dashboard_geospatial.csv' (HEADER, DELIMITER ',');

-- 2. Exporting Promo ROI Data
COPY (
    SELECT 
        CASE 
            WHEN promo_code IS NULL THEN 'No Promo' 
            ELSE 'Used Promo' 
        END AS promo_status,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(total_amount_idr) AS total_revenue,
        SUM(COALESCE(discount_amount_idr, 0)) AS total_discount_given,
        CAST(SUM(total_amount_idr) / COUNT(DISTINCT order_id) AS BIGINT) AS average_order_value
    FROM orders
    WHERE order_status NOT IN ('cancelled', 'returned')
    GROUP BY promo_status
) TO 'dashboard_promo_roi.csv' (HEADER, DELIMITER ',');

-- 3. Exporting Courier Evaluation Data
COPY (
    SELECT 
        courier,
        COUNT(order_id) AS total_orders,
        SUM(CASE WHEN order_status = 'returned' THEN 1 ELSE 0 END) AS returned_orders,
        SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
        ROUND(SUM(CASE WHEN order_status = 'returned' THEN 1.0 ELSE 0 END) / COUNT(order_id) * 100, 2) AS return_rate_pct,
        ROUND(SUM(CASE WHEN order_status = 'cancelled' THEN 1.0 ELSE 0 END) / COUNT(order_id) * 100, 2) AS cancel_rate_pct
    FROM orders
    GROUP BY courier
    ORDER BY return_rate_pct DESC
) TO 'dashboard_courier_eval.csv' (HEADER, DELIMITER ',');

-- 4. Exporting RFM Segmentation Data
COPY (
    WITH rfm_base AS (
        SELECT 
            c.email,
            CAST((SELECT MAX(order_date) FROM orders) AS DATE) - CAST(MAX(o.order_date) AS DATE) AS recency_days,
            COUNT(DISTINCT o.order_id) AS frequency,
            SUM(o.total_amount_idr) AS monetary_value
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY o.customer_id, c.email
    )
    SELECT 
        email,
        recency_days,
        frequency, 
        monetary_value,
        CASE 
            WHEN recency_days <= 60 AND frequency >= 5 AND monetary_value >= 4000000 THEN '🏆 Champions'
            WHEN recency_days > 60 AND (frequency >= 5 OR monetary_value >= 4000000) THEN '⚠️ At Risk (VIPs)'
            WHEN recency_days <= 30 AND frequency < 3 THEN '👋 New / Promising'
            WHEN recency_days >= 120 THEN '💤 Lost / Inactive'
            ELSE '🛒 Regulars'
        END AS customer_segment
    FROM rfm_base
) TO 'dashboard_rfm_segments.csv' (HEADER, DELIMITER ',');

Running query in 'DuckDBPyConnection'

Count
787


**✅ Phase 2 (Exploratory Data Analysis) Complete.** 